# Estudio 03 — IA Conexionista: Perceptrón y TinyML en Arduino
**Módulo 5: Embedded Systems & TinyML**

---

## ¿Qué es la IA Conexionista?

La IA Conexionista (también llamada conexionismo o sub-simbólica) modela la inteligencia mediante redes de unidades simples (neuronas artificiales) que aprenden de los datos.

### El Perceptrón (Rosenblatt, 1958)

El perceptrón es la unidad fundamental de una red neuronal:

**Fórmula:**
```
  z = w₁·x₁ + w₂·x₂ + ... + wₙ·xₙ + b
  ŷ = 1  si z ≥ 0
  ŷ = 0  si z < 0
```

### Regla de aprendizaje de Rosenblatt:
1. Inicializar pesos (W) y sesgo (b) aleatoriamente
2. Para cada muestra de entrenamiento (x, y):
   - Calcular predicción ŷ
   - Error = y - ŷ
   - Si error ≠ 0: W ← W + lr · error · x
                    b ← b + lr · error
3. Repetir hasta convergencia (sin errores)

### Limitación:
El perceptrón solo puede separar puntos **linealmente separables**. XOR no es linealmente separable → requiere una red multicapa (MLP).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..')))
from libreria_modulo5 import *
import numpy as np

In [ ]:
configurar_reproducibilidad(42)

## Entrenar Perceptrón para compuertas lógicas

### 1. Compuerta AND

In [ ]:
res_and = perceptron_compuerta('AND', lr=0.1, epochs=20, verbose=True)

### 2. Compuerta OR

In [ ]:
res_or = perceptron_compuerta('OR', lr=0.1, epochs=20, verbose=True)

### 3. Compuerta NOT

In [ ]:
res_not = perceptron_compuerta('NOT', lr=0.1, epochs=20, verbose=True)

## Visualización de Convergencia

Graficamos cómo disminuyen los errores durante el entrenamiento del perceptrón AND.

In [ ]:
graficar_convergencia(res_and['history'], 'Convergencia — Perceptrón AND')

## Frontera de Decisión

Visualizamos la recta que separa las clases 0 y 1 (AND). El perceptrón aprende una línea recta que separa los puntos.

In [ ]:
X_and = np.array([[0,0],[0,1],[1,0],[1,1]])
y_and = np.array([0,0,0,1])

graficar_frontera_decision(
    res_and['W'], res_and['b'],
    X_and, y_and,
    'Frontera de Decisión — Perceptrón AND'
)

## XOR: El problema no linealmente separable

XOR no es linealmente separable — no existe una línea recta que separe (0,0) y (1,1) de (0,1) y (1,0).

Intentemos entrenar un perceptrón para XOR:

In [ ]:
print("=== Intentando entrenar perceptrón para XOR ===")

X_xor = np.array([[0,0],[0,1],[1,0],[1,1]])
y_xor = np.array([0,1,1,0])

res_xor = perceptron_entrenar(X_xor, y_xor, lr=0.1, epochs=50, verbose=True)

print(f"\nPrecisión final: {res_xor['accuracy']:.0f}%")
print(f"Pesos: {np.round(res_xor['W'], 3)}, Bias: {round(res_xor['b'], 3)}")

In [ ]:
preds = perceptron_binario(res_xor['W'], res_xor['b'], X_xor)
print("Predicciones XOR vs reales:")
print("  A   B |  ŷ   y")
print("  ------+--------")
for (a, b), pred, real in zip(X_xor, preds, y_xor):
    ok = '✓' if pred == real else '✗'
    print(f"  {a}   {b} |  {pred}   {real}  {ok}")

### ¿Por qué falla XOR?

El perceptrón solo puede aprender **fronteras lineales** (rectas en 2D, planos en 3D, hiperplanos en nD).

**Visualización**: Los puntos de XOR están en las esquinas opuestas del cuadrado:
```
  (0,1) → 1          (1,1) → 0
        ┌──────────┐
        │          │
        │    ???   │  ← No hay recta que separe
        │          │
        └──────────┘
  (0,0) → 0          (1,0) → 1
```

**Solución**: Usar un **Perceptrón Multicapa (MLP)** con al menos una capa oculta. El MLP aprende fronteras no lineales combinando múltiples perceptrones.

> **Conclusión**: Compuertas AND, OR, NOT, NAND, NOR son linealmente separables. XOR y XNOR no lo son — necesitan redes más profundas.

In [ ]:
graficar_frontera_decision(
    res_xor['W'], res_xor['b'],
    X_xor, y_xor,
    'XOR — Frontera de Decisión (falla: no linealmente separable)'
)

## Exportar Perceptrón a C++ para Arduino

El perceptrón entrenado puede ejecutarse en un microcontrolador. La función `exportar_perceptron_a_c()` genera código C++ listo para copiar al Arduino IDE.

In [ ]:
codigo_and = exportar_perceptron_a_c(
    res_and['W'], res_and['b'],
    nombre_funcion='predecir_and',
    nombres_entrada=['x1', 'x2'],
    tipo='int'
)
print(codigo_and)

In [ ]:
# También podemos exportar como float para mayor precisión
codigo_or_float = exportar_perceptron_a_c(
    res_or['W'], res_or['b'],
    nombre_funcion='predecir_or',
    nombres_entrada=['sensor1', 'sensor2'],
    tipo='float'
)
print(codigo_or_float)

## Exportar sketch .ino completo

In [ ]:
sketch = exportar_modelo_arduino_ino(
    res_and['W'], res_and['b'],
    nombre='AND_perceptron',
    tipo='int'
)
print(sketch)

## Concepto TinyML

**TinyML** es la intersección del Machine Learning y los sistemas embebidos de bajo consumo.

### Flujo de trabajo:
```
  ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐
  │ 1. TRAIN │    │ 2. EXPORT│    │ 3. COMPIL│    │ 4. DEPLOY│
  │ Python   │───→│  C/C++   │───→│  Arduino │───→│  MCU     │
  │ (PC)     │    │  código  │    │  IDE     │    │  (UNO)   │
  └──────────┘    └──────────┘    └──────────┘    └──────────┘
```

### ¿Por qué TinyML?
- **Bajo consumo**: µA en lugar de Watts (vs. GPU)
- **Latencia cero**: inferencia local, sin red
- **Privacidad**: datos nunca salen del dispositivo
- **Costo**: MCU cuesta $1-$10 vs. $10,000+ GPU

### Aplicaciones:
- Detección de palabras clave ("Hey Google" en un MCU)
- Clasificación de gestos con IMU
- Mantenimiento predictivo (vibraciones)
- Agricultura inteligente (humedad del suelo)

## Ejercicio #TODO — Perceptrón en datos de sensores

**Objetivo**: Entrenar un perceptrón para clasificar si un sensor debe activar una alarma.

### Contexto:
Tienes un sensor de temperatura y un sensor de humedad en un invernadero. La alarma debe activarse cuando:
- Temperatura > 30°C **Y** Humedad < 40%  (condición de riesgo)
- O cuando Temperatura > 35°C (independientemente de la humedad)

### Datos normalizados:
Usa `mapear_0_1()` para convertir lecturas raw a [0,1]:

```python
# Datos raw simulados: (temp °C, humedad %)
datos_raw = [
    (25, 60),   # Normal
    (28, 55),   # Normal
    (32, 35),   # Riesgo (temp > 30 Y humedad < 40)
    (30, 50),   # Normal (justo en el límite)
    (37, 45),   # Riesgo (temp > 35)
    (22, 70),   # Normal
    (35, 30),   # Riesgo (temp >= 35 Y humedad < 40)
    (40, 60),   # Riesgo (temp > 35)
]

# Etiquetas (1 = alarma, 0 = normal)
y_alarma = [0, 0, 1, 0, 1, 0, 1, 1]
```

### Pasos:
1. Normaliza los datos con `mapear_0_1(temp, 20, 45)` y `mapear_0_1(humedad, 20, 80)`
2. Entrena un perceptrón con `perceptron_entrenar(X, y, lr=0.1, epochs=30)`
3. Grafica la convergencia
4. Grafica la frontera de decisión
5. Exporta el modelo a C++ con `exportar_perceptron_a_c()`
6. Genera un sketch .ino completo con `exportar_modelo_arduino_ino()`

> ⚠️ **Reto extra**: Conecta un Arduino real con sensor DHT11 y ejecuta el perceptrón en tiempo real.